# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 09

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. Аналитик выполнил две строчки кода:
```python
s1 = df['balance']
s2 = df[['balance']]
```
В какой переменной (`s1` или `s2`) получен одномерный столбец (`pd.Series`), а в какой — двумерная таблица (`pd.DataFrame`) из одной колонки? В чем их главное отличие при обращении к методам?

**Ответ:**
* `s1 = df['balance']` возвращает одномерную структуру `pd.Series` (вектор значений).
* `s2 = df[['balance']]` (список в двойных квадратных скобках) возвращает двумерную таблицу `pd.DataFrame` из одной колонки.
К `Series` обращаются как к одномерному вектору, а к `DataFrame` применимы табличные методы (выборка нескольких колонок, отображение в виде таблицы).

---

#### 1.2. Аналитик вызвал `df.info()` для таблицы игровых сессий из 500 строк и увидел запись:
`client_rating: 425 non-null float64`.
Сколько сессий в таблице остались без клиентской оценки (содержат `NaN` / `None`), и какой метод Pandas позволяет подсчитать количество пропусков во всех столбцах?

**Ответ:**
Без оценки остались **75 сессий** (500 - 425 = 75). Для подсчета точного количества пропусков в каждом столбце используется метод `df.isna().sum()` (или `df.isnull().sum()`).

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["vip_client"] = (df["balance"] >= 2000) & (df["city"] == "Москва")
```

**Ответ:**
Формирует логический флаг VIP-клиента: баланс лицевого счета от 2000 рублей при условии проживания в Москве.

---

#### 1.4. Аналитик выполнил расчет средней суммы оплат по играм:
```python
report = df.groupby('game_title')['total_paid'].mean()
```
Сколько строк получится в объекте `report` (столько же, сколько всех сессий, или столько, сколько уникальных игр)? Что стало индексом строк в полученном объекте?

**Ответ:**
В объекте `report` строк будет ровно столько, сколько **уникальных игр** в таблице (строки свернулись по уникальным значениям `game_title`). Новым индексом строк стали сами названия игр (`game_title`).

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `bar_orders.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path("../variant_09")
df_audit = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Средняя цена товара в баре: 335.06")

Количество строк в таблице: 350
Столбцы датафрейма: ['order_id', 'member_id', 'item_name', 'category', 'order_date', 'quantity', 'item_price', 'total_amount']
Средняя цена товара в баре: 335.06

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_09")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,86400,1.36
1,Горячее,33090,1.49
2,Напитки,32120,1.61
3,Снеки,15880,1.51


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `members` поле `member_id` объявлено как `PRIMARY KEY` (первичный ключ).
Что произойдет, если система попытается добавить нового клиента с уже существующим номером `member_id = 1` или со значением `NULL`? Почему СУБД вернет ошибку?

**Ответ:**
СУБД заблокирует операцию и вернет ошибку нарушения ограничения целостности (`PRIMARY KEY / UNIQUE / NOT NULL constraint failed`). Значение первичного ключа обязано быть уникальным для каждой записи и не может быть пустым (`NULL`), так как оно однозначно идентифицирует строку.

---

#### 3.2. Разработчик через скрипт на Python выполнил добавление новых компьютеров командой `cursor.execute("INSERT INTO hardware_pcs ...")`, но после закрытия программы обнаружил, что компьютеры не сохранились в файле базы данных.
Какую обязательную команду для объекта соединения (`conn`) он забыл вызвать и почему?

**Ответ:**
Он забыл вызвать команду фиксации транзакции `conn.commit()`. В библиотеке `sqlite3` изменения, внесенные через курсор, по умолчанию находятся во временной транзакции и не сохраняются на диске без явного `commit()`.

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT city, COUNT(*) AS members_cnt, AVG(balance) AS avg_bal FROM members GROUP BY city ORDER BY members_cnt DESC;
```

**Ответ:**
Запрос группирует клиентов по городам (`city`) и вычисляет число клиентов (`COUNT(*)`) и их средний баланс (`AVG(balance)`), сортируя по убыванию числа клиентов.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_09")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT pc_id, pc_number, zone_name, hourly_rate, gpu, ram_gb, status
FROM hardware_pcs
WHERE zone_name IN ('Standard', 'Bootcamp') AND hourly_rate <= 250
ORDER BY hourly_rate ASC, pc_id ASC;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,pc_id,pc_number,zone_name,hourly_rate,gpu,ram_gb,status
0,3,PC-03,Standard,130.0,RTX 2060,16,Active
1,7,PC-07,Standard,130.0,GTX 1660 Super,16,Active
2,9,PC-09,Standard,130.0,GTX 1660 Super,16,Active
3,10,PC-10,Standard,130.0,RTX 2060,16,Active
4,1,PC-01,Standard,150.0,RTX 3050,16,Active
5,2,PC-02,Standard,150.0,GTX 1660 Super,16,Active
6,4,PC-04,Standard,150.0,GTX 1660 Super,16,Active
7,5,PC-05,Standard,150.0,GTX 1660 Super,16,Active
8,6,PC-06,Standard,150.0,RTX 3050,16,Active
9,13,PC-13,Standard,150.0,RTX 2060,16,Active


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT status, COUNT(*) AS pcs_count, ROUND(AVG(ram_gb), 2) AS avg_ram
FROM hardware_pcs
GROUP BY status
HAVING COUNT(*) >= 3
ORDER BY pcs_count DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,status,pcs_count,avg_ram
0,Active,26,28.31


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, m.loyalty_tier, COUNT(b.order_id) AS orders_count, ROUND(SUM(b.total_amount), 2) AS total_spent
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
WHERE m.loyalty_tier IN ('Gold', 'Platinum')
GROUP BY m.member_id, m.nickname, m.loyalty_tier
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,loyalty_tier,orders_count,total_spent
0,109,Shadow_725,Gold,6,4465.0
1,106,Flash_140,Gold,6,4430.0
2,61,Cobra_724,Gold,5,3490.0
3,55,Rogue_453,Platinum,5,2830.0
4,105,CyberKnight_396,Gold,4,2785.0
5,127,Warlock_936,Gold,5,2770.0
6,120,Dragon_420,Gold,5,2600.0
7,116,Echo_146,Gold,5,2580.0
8,54,Zero_186,Platinum,5,2560.0
9,85,Mystic_376,Gold,2,2280.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid
FROM game_sessions
WHERE total_paid > (SELECT AVG(total_paid) FROM game_sessions WHERE discount_promo = 1)
ORDER BY total_paid DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,session_id,member_id,game_title,duration_hours,total_paid
0,20080,12,Warzone,6.0,2520.0
1,20212,77,Atomic Heart,6.0,2400.0
2,20323,83,Dota 2,6.0,2400.0
3,20021,116,Warzone,6.0,2160.0
4,20251,33,PUBG,5.0,2000.0
5,20111,60,PUBG,4.0,1680.0
6,20043,63,Apex Legends,6.0,1620.0
7,20242,94,PUBG,6.0,1620.0
8,20342,109,Valorant,6.0,1620.0
9,20449,74,Cyberpunk 2077,6.0,1620.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Выручка клуба по играм

**Найденные ошибки:**
1. Агрегатная функция в `WHERE`: условие `WHERE SUM(total_paid) > 15000` недопустимо, фильтрация агрегированных значений должна выполняться в `HAVING`.

In [9]:
query_51_fixed = '''
SELECT 
    game_title,
    COUNT(*) AS cnt,
    SUM(total_paid) AS total_revenue
FROM game_sessions
GROUP BY game_title
HAVING SUM(total_paid) > 15000
ORDER BY total_revenue DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,game_title,cnt,total_revenue
0,Dota 2,56,28660.0
1,PUBG,57,28163.0
2,Cyberpunk 2077,54,24416.0
3,Rust,56,23682.0
4,Counter-Strike 2,53,23682.0
5,Valorant,49,23256.0
6,Atomic Heart,45,22343.0
7,GTA V,49,21615.0
8,Warzone,39,19126.0
9,Apex Legends,42,16970.0


#### 5.2. Траты клиентов на игровое время

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает идентификатор клиента с номером компьютера вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: фильтрация по `SUM(s.total_paid) > 1000` должна выполняться в секции `HAVING`, а не в `WHERE`.

In [10]:
query_52_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname
HAVING SUM(s.total_paid) > 1000
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,member_id,nickname,total_spent
0,109,Shadow_725,5516.0
1,79,Paladin_300,4555.0
2,77,Blaze_578,4488.0
3,116,Echo_146,4370.0
4,83,Hunter_634,4070.0
5,90,Lynx_109,4065.0
6,63,Rogue_889,3937.0
7,8,Turbo_249,3920.0
8,74,Phoenix_642,3688.0
9,105,CyberKnight_396,3640.0


#### 5.3. Анализ активности и расходов клиентов

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает идентификатор клиента с номером компьютера вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: условие `WHERE SUM(s.total_paid) > 1200` должно быть перенесено в блок `HAVING`.
3. Неполный `GROUP BY`: столбцы `m.nickname` и `m.city` присутствуют в `SELECT`, но пропущены в списке группировки.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    COUNT(s.session_id) AS cnt,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING SUM(s.total_paid) > 1200
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,cnt,total_spent
0,109,Shadow_725,Москва,8,5516.0
1,79,Paladin_300,Казань,8,4555.0
2,77,Blaze_578,Санкт-Петербург,5,4488.0
3,116,Echo_146,Санкт-Петербург,7,4370.0
4,83,Hunter_634,Москва,5,4070.0
5,90,Lynx_109,Санкт-Петербург,8,4065.0
6,63,Rogue_889,Москва,7,3937.0
7,8,Turbo_249,Санкт-Петербург,7,3920.0
8,74,Phoenix_642,Москва,6,3688.0
9,105,CyberKnight_396,Москва,7,3640.0
